# Homologación de datos electorales — Cárdenas, Tabasco
PELO 2023-2024: Gubernatura, Diputaciones y Ayuntamientos

Objetivo: consolidar los 7 archivos .xlsx (sección electoral × casilla)
en una sola tabla normalizada lista para el geovisor.

In [1]:
import pandas as pd
from pathlib import Path


pd.set_option('display.max_columns', None)

In [2]:
TABLAS    = Path("../datos/insumos/tablas")
PROCESADOS = Path("../datos/procesados")
PROCESADOS.mkdir(exist_ok=True)

## 1. Exploración rápida de cada archivo

In [3]:
archivos = sorted(TABLAS.glob("*.xlsx"))

for f in archivos:
    df = pd.read_excel(f)
    print(f"{'='*60}")
    print(f"Archivo : {f.name}")
    print(f"Filas   : {len(df)}  |  Columnas: {len(df.columns)}")
    print(f"Cols    : {list(df.columns)}")
    print()

Archivo : cardenas_desglose_ayuntamientos_PELO2023-2024.xlsx
Filas   : 321  |  Columnas: 20
Cols    : ['#', 'Sección Electoral', 'Casilla', 'PRI', 'PRD', 'PVEM', 'PT', 'MC', 'MORENA', ' C. IND. JUAN CARLOS GUZMAN CORREA', 'PVEM Y MORENA', 'TOTAL \nPVEM Y MORENA', 'NÚMERO DE VOTOS VÁLIDOS', 'NÚMERO DE VOTOS CANDIDATURAS NO REGISTRADAS', 'NÚMERO DE VOTOS NULOS', 'TOTAL DE VOTOS', 'LISTA NOMINAL', '% PARTICIPACIÓN CIUDADANA', 'ANULADA', 'MODIFICADA']

Archivo : cardenas_desglose_dt01_gubernatura.xlsx
Filas   : 160  |  Columnas: 24
Cols    : ['#', 'SECCIÓN ELECTORAL', 'CASILLA', 'PAN', 'PRI', 'PRD', 'PVEM', 'PT', 'MC', 'MORENA', 'PVEM, PT Y MORENA', 'PVEM Y PT', 'PVEM Y MORENA', 'PT Y MORENA', 'TOTAL\nPVEM, PT Y MORENA', 'PAN Y PRI', 'TOTAL\nPAN Y PRI', 'NÚMERO DE VOTOS VÁLIDOS', 'NÚMERO DE VOTOS CANDIDATURAS NO REGISTRADAS', 'NÚMERO DE VOTOS NULOS', 'TOTAL DE VOTOS', 'LISTA NOMINAL', '% PARTICIPACIÓN CIUDADANA', 'ANULADA']

Archivo : cardenas_desglose_dt02_gubernatura.xlsx
Filas   : 138  

## 2. Esquema objetivo

 Columnas comunes a todos los archivos (nombres normalizados):

 seccion        — sección electoral (str, 4 dígitos con cero a la izquierda)
 casilla        — nombre de la casilla (BÁSICA, CONTIGUA 01, VOTO ANTICIPADO…)
 tipo_eleccion  — 'gubernatura' | 'diputaciones' | 'ayuntamiento'
 distrito       — 'DT-01'…'DT-03' | 'DTO-01'…'DTO-03' | None (ayuntamiento cubre los 3)

 Votos por partido (pueden ser NaN si el partido no participó en esa elección):
 PAN, PRI, PRD, PVEM, PT, MC, MORENA, CAND_IND

 Totales de coalición (cuando aplica):
 total_pvem_pt_morena   — TOTAL PVEM, PT Y MORENA  (solo gubernatura)
 total_pan_pri          — TOTAL PAN Y PRI           (solo gubernatura)
 total_pvem_morena      — TOTAL PVEM Y MORENA       (gubernatura + dto01)

 Resumen de votación:
 votos_validos, cand_no_reg, votos_nulos, total_votos, lista_nominal, participacion

 Banderas (cuando aplica):
 anulada, modificada

## 3. Funciones de normalización

In [4]:
# Mapeo de columnas crudas → nombre normalizado.
# Clave: nombre crudo (tal como aparece en el xlsx, ignorando mayúsculas/minúsculas).
# Valor: nombre objetivo en el esquema unificado.

RENAME_MAP = {
    # sección
    "sección electoral"                               : "seccion",
    "seccion electoral"                               : "seccion",  # sin tilde
    # casilla
    "casilla"                                         : "casilla",
    # partidos
    "pan"                                             : "PAN",
    "pri"                                             : "PRI",
    "prd"                                             : "PRD",
    "pvem"                                            : "PVEM",
    "pt"                                              : "PT",
    "mc"                                              : "MC",
    "morena"                                          : "MORENA",
    "c. ind. juan carlos guzman correa"               : "CAND_IND",
    # coaliciones — solo los TOTALES (las sub-columnas se descartan)
    "total pvem, pt y morena"                         : "total_pvem_pt_morena",
    "total pvem y morena"                             : "total_pvem_morena",
    "total pan y pri"                                 : "total_pan_pri",
    # resumen
    "número de votos válidos"                         : "votos_validos",
    "numero de votos validos"                         : "votos_validos",
    "número de votos candidaturas no registradas"     : "cand_no_reg",
    "numero de votos candidaturas no registradas"     : "cand_no_reg",
    "número de votos a candidaturas no registradas"   : "cand_no_reg",
    "numero de votos a candidaturas no registradas"   : "cand_no_reg",
    "número de votos nulos"                           : "votos_nulos",
    "numero de votos nulos"                           : "votos_nulos",
    "total de votos"                                  : "total_votos",
    "lista nominal"                                   : "lista_nominal",
    "% participación ciudadana"                       : "participacion",
    "% participacion ciudadana"                       : "participacion",
    "participación ciudadana"                         : "participacion",
    "participacion ciudadana"                         : "participacion",
    # banderas
    "anulada"                                         : "anulada",
    "modificada"                                      : "modificada",
}

# Sub-columnas de coalición que se descartan (son parciales, no totales)
COLS_DESCARTAR = {
    "pvem, pt y morena", "pvem y pt", "pvem y morena", "pt y morena",
    "pan y pri",
    "pvem y morena",       # sub-columna en gubernatura (distinto de TOTAL PVEM Y MORENA)
}

# Columnas del esquema final (en orden)
COLS_FINALES = [
    "seccion", "casilla", "tipo_eleccion", "distrito",
    "PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND",
    "total_pvem_pt_morena", "total_pvem_morena", "total_pan_pri",
    "votos_validos", "cand_no_reg", "votos_nulos", "total_votos",
    "lista_nominal", "participacion",
    "anulada", "modificada",
]

In [5]:
def normalizar_archivo(ruta, tipo_eleccion, distrito=None):
    """
    Lee un .xlsx, normaliza columnas y filtra filas de totales/resumen.

    Parámetros
    ----------
    ruta          : Path al archivo .xlsx
    tipo_eleccion : 'gubernatura' | 'diputaciones' | 'ayuntamiento'
    distrito      : str o None — e.g. 'DT-01', 'DTO-02'

    Retorna
    -------
    DataFrame normalizado con las columnas de COLS_FINALES que existan.
    """
    df = pd.read_excel(ruta)

    # 1. Eliminar columna índice artificial (#, Unnamed)
    df = df.drop(columns=[c for c in df.columns
                           if str(c).strip() in ("#", "Unnamed: 0")], errors="ignore")

    # 2. Clave normalizada: minúsculas + colapsar cualquier espacio/salto de línea
    def _key(c):
        return " ".join(str(c).strip().lower().split())

    # 3. Renombrar usando el mapa
    rename = {
        col: RENAME_MAP[_key(col)]
        for col in df.columns
        if _key(col) in RENAME_MAP
    }
    df = df.rename(columns=rename)

    # 4. Descartar sub-columnas de coalición que no son totales
    df = df.drop(
        columns=[c for c in df.columns if _key(c) in COLS_DESCARTAR],
        errors="ignore"
    )

    # 5. Filtrar filas de totales/resumen
    #    - Se conservan: filas con casilla definida (incluye VOTO ANTICIPADO)
    #    - Se eliminan: filas donde casilla es NaN (totales al pie de la tabla)
    df = df[df["casilla"].notna()].copy()

    # 6. Normalizar sección: rellenar NaN de VOTO ANTICIPADO con "0000"
    #    y asegurar formato de 4 dígitos con cero a la izquierda
    df["seccion"] = (
        df["seccion"]
        .fillna(0)
        .astype(float)
        .astype(int)
        .astype(str)
        .str.zfill(4)
    )

    # 7. Agregar columnas de metadatos
    df["tipo_eleccion"] = tipo_eleccion
    df["distrito"]      = distrito

    # 8. Reindexar al esquema final (columnas ausentes quedan como NaN)
    df = df.reindex(columns=COLS_FINALES)

    return df.reset_index(drop=True)

## 4. Carga y normalización de cada archivo

In [6]:
FUENTES = [
    # (archivo,                                        tipo_eleccion,    distrito )
    ("cardenas_desglose_dt01_gubernatura.xlsx",        "gubernatura",    "DT-01"  ),
    ("cardenas_desglose_dt02_gubernatura.xlsx",        "gubernatura",    "DT-02"  ),
    ("cardenas_desglose_dt03_gubernatura.xlsx",        "gubernatura",    "DT-03"  ),
    ("cardenas_desglose_dto01_diputaciones.xlsx",      "diputaciones",   "DTO-01" ),
    ("cardenas_desglose_dto02_diputaciones.xlsx",      "diputaciones",   "DTO-02" ),
    ("cardenas_desglose_dto03_diputaciones.xlsx",      "diputaciones",   "DTO-03" ),
    ("cardenas_desglose_ayuntamientos_PELO2023-2024.xlsx", "ayuntamiento", None   ),
]

partes = []
for nombre, tipo, distrito in FUENTES:
    df_parte = normalizar_archivo(TABLAS / nombre, tipo, distrito)
    print(f"{nombre[:50]:<52}  →  {len(df_parte):>3} filas")
    partes.append(df_parte)

resultados = pd.concat(partes, ignore_index=True)
print(f"\nTotal consolidado: {len(resultados)} filas × {len(resultados.columns)} columnas")

cardenas_desglose_dt01_gubernatura.xlsx               →  159 filas
cardenas_desglose_dt02_gubernatura.xlsx               →  137 filas
cardenas_desglose_dt03_gubernatura.xlsx               →  130 filas
cardenas_desglose_dto01_diputaciones.xlsx             →  160 filas
cardenas_desglose_dto02_diputaciones.xlsx             →  136 filas
cardenas_desglose_dto03_diputaciones.xlsx             →  131 filas
cardenas_desglose_ayuntamientos_PELO2023-2024.xlsx    →  320 filas

Total consolidado: 1173 filas × 23 columnas


## 5. Verificación de calidad

In [7]:
resultados.head(3)

,seccion,casilla,tipo_eleccion,distrito,PAN,PRI,PRD,PVEM,PT,MC,MORENA,CAND_IND,total_pvem_pt_morena,total_pvem_morena,total_pan_pri,votos_validos,cand_no_reg,votos_nulos,total_votos,lista_nominal,participacion,anulada,modificada
0,0071,BÁSICA,gubernatura,DT-01,12.0,5.0,40.0,12.0,14.0,20.0,244.0,NaN,288.0,NaN,17.0,365,0,9,374,660,0.566666,NO,NaN
1,0071,CONTIGUA 01,gubernatura,DT-01,6.0,4.0,44.0,5.0,10.0,16.0,231.0,NaN,259.0,NaN,11.0,330,0,5,335,660,0.507575,NO,NaN
2,0071,CONTIGUA 02,gubernatura,DT-01,6.0,9.0,35.0,17.0,13.0,20.0,227.0,NaN,284.0,NaN,15.0,354,0,13,367,660,0.556060,NO,NaN


In [8]:
resultados[resultados["seccion"]=='0071']

,seccion,casilla,tipo_eleccion,distrito,PAN,PRI,PRD,PVEM,PT,MC,MORENA,CAND_IND,total_pvem_pt_morena,total_pvem_morena,total_pan_pri,votos_validos,cand_no_reg,votos_nulos,total_votos,lista_nominal,participacion,anulada,modificada
0,0071,BÁSICA,gubernatura,DT-01,12.0,5.0,40.0,12.0,14.0,20.0,244.0,NaN,288.0,NaN,17.0,365,0,9,374,660,0.566666,NO,NaN
1,0071,CONTIGUA 01,gubernatura,DT-01,6.0,4.0,44.0,5.0,10.0,16.0,231.0,NaN,259.0,NaN,11.0,330,0,5,335,660,0.507575,NO,NaN
2,0071,CONTIGUA 02,gubernatura,DT-01,6.0,9.0,35.0,17.0,13.0,20.0,227.0,NaN,284.0,NaN,15.0,354,0,13,367,660,0.556060,NO,NaN
3,0071,CONTIGUA 03,gubernatura,DT-01,9.0,5.0,37.0,9.0,17.0,11.0,221.0,NaN,274.0,NaN,14.0,336,1,17,354,660,0.536363,NO,NaN
4,0071,CONTIGUA 04,gubernatura,DT-01,7.0,9.0,53.0,34.0,9.0,25.0,210.0,NaN,253.0,NaN,16.0,347,0,13,360,660,0.545454,NO,NaN
5,0071,CONTIGUA 05,gubernatura,DT-01,10.0,2.0,38.0,8.0,12.0,17.0,249.0,NaN,288.0,NaN,13.0,356,0,5,361,660,0.546969,NO,NaN
6,0071,CONTIGUA 06,gubernatura,DT-01,15.0,6.0,44.0,19.0,12.0,20.0,247.0,NaN,290.0,NaN,22.0,376,0,9,385,659,0.584218,NO,NaN
427,0071,BÁSICA,diputaciones,DTO-01,14.0,12.0,51.0,24.0,11.0,26.0,212.0,NaN,NaN,242.0,NaN,356,0,18,374,660,0.566666,NaN,NaN
428,0071,CONTIGUA 01,diputaciones,DTO-01,11.0,12.0,43.0,13.0,6.0,21.0,209.0,NaN,NaN,227.0,NaN,320,0,15,335,660,0.507575,NaN,NaN
429,0071,CONTIGUA 02,diputaciones,DTO-01,3.0,12.0,54.0,19.0,9.0,23.0,208.0,NaN,NaN,236.0,NaN,337,1,29,367,660,0.556060,NaN,NaN


In [9]:
# Distribución por tipo de elección y distrito
resultados.groupby(["tipo_eleccion", "distrito"]).size().rename("casillas")

tipo_eleccion  distrito
diputaciones   DTO-01      160
               DTO-02      136
               DTO-03      131
gubernatura    DT-01       159
               DT-02       137
               DT-03       130
Name: casillas, dtype: int64

In [10]:
# Nulos por columna (solo columnas con al menos un nulo)
nulos = resultados.isna().sum()
nulos[nulos > 0]

distrito                320
PAN                     322
PRI                       3
PRD                       3
PVEM                      3
PT                        3
MC                        3
MORENA                    3
CAND_IND                854
total_pvem_pt_morena    747
total_pvem_morena       693
total_pan_pri           747
participacion            14
anulada                 427
modificada              853
dtype: int64

In [11]:
# Secciones únicas por tipo de elección
resultados.groupby("tipo_eleccion")["seccion"].nunique().rename("secciones_unicas")

tipo_eleccion
ayuntamiento    123
diputaciones    159
gubernatura     159
Name: secciones_unicas, dtype: int64

In [12]:
# Verificar que total_votos == suma de partidos (donde aplica)
COLS_PARTIDOS = ["PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND"]

# Para gubernatura, los votos de PVEM/PT/MORENA individuales + coalición pueden solaparse.
# Esta verificación es orientativa para diputaciones y ayuntamiento.
mask = resultados["tipo_eleccion"].isin(["diputaciones", "ayuntamiento"])
df_check = resultados[mask].copy()

df_check["suma_partidos"] = df_check[COLS_PARTIDOS].sum(axis=1, min_count=1)
df_check["diff"] = df_check["votos_validos"] - df_check["suma_partidos"]

print("Diferencia votos_validos − suma_partidos:")
print(df_check["diff"].describe().round(2))
print(f"\nFilas con diferencia > 1: {(df_check['diff'].abs() > 1).sum()}")

Diferencia votos_validos − suma_partidos:
count    745.00
mean       2.69
std        3.77
min        0.00
25%        0.00
50%        1.00
75%        5.00
max       45.00
Name: diff, dtype: float64

Filas con diferencia > 1: 349


## 6. Agregación a nivel de sección

Los datos del geovisor se consumen por sección electoral (no por casilla).  
Colapsamos las filas sumando los votos y recalculamos los campos derivados:  
`participacion = total_votos / lista_nominal`

Se generan **tres archivos independientes**, uno por tipo de elección.

In [13]:
# Columnas que se agregan por suma directa
COLS_SUMA = [
    "PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND",
    "total_pvem_pt_morena", "total_pvem_morena", "total_pan_pri",
    "votos_validos", "cand_no_reg", "votos_nulos", "total_votos",
    "lista_nominal",
]

# Esquema de columnas para los archivos de sección
ESQUEMA_SECCION = [
    "seccion", "tipo_eleccion", "distrito",
    "PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND",
    "total_pvem_pt_morena", "total_pvem_morena", "total_pan_pri",
    "votos_validos", "cand_no_reg", "votos_nulos", "total_votos",
    "lista_nominal", "participacion",
]


def agregar_por_seccion(df):
    """
    Colapsa las filas de casilla al nivel de sección electoral.

    - Agrupa por seccion + distrito (distrito=None para ayuntamiento).
    - Suma todos los votos (partidos, coaliciones, totales de acta).
    - Recalcula participacion = total_votos / lista_nominal post-agregación.
    """
    tipo = df["tipo_eleccion"].iloc[0]
    cols_suma = [c for c in COLS_SUMA if c in df.columns]

    agg = (
        df
        .groupby(["seccion", "distrito"], dropna=False)[cols_suma]
        .sum(min_count=1)
        .reset_index()
    )

    # Campos recalculados
    agg["tipo_eleccion"] = tipo
    agg["participacion"] = agg["total_votos"] / agg["lista_nominal"]

    return agg.reindex(columns=ESQUEMA_SECCION).reset_index(drop=True)


secciones = {}
for tipo in ["gubernatura", "diputaciones", "ayuntamiento"]:
    df_sec = agregar_por_seccion(resultados[resultados["tipo_eleccion"] == tipo])
    secciones[tipo] = df_sec
    print(f"{tipo:<15}  →  {len(df_sec):>3} secciones × {len(df_sec.columns)} cols")

print(f"\nTotal secciones: {sum(len(d) for d in secciones.values())}")

gubernatura      →  159 secciones × 20 cols
diputaciones     →  160 secciones × 20 cols
ayuntamiento     →  123 secciones × 20 cols

Total secciones: 442


In [14]:
# Verificación: votos_validos debe ≈ suma de partidos
for tipo, df_sec in secciones.items():
    cols_partidos = [c for c in ["PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND"]
                     if c in df_sec.columns]
    df_sec["_suma_partidos"] = df_sec[cols_partidos].sum(axis=1, min_count=1)
    df_sec["_diff"] = df_sec["votos_validos"] - df_sec["_suma_partidos"]
    max_diff = df_sec["_diff"].abs().max()
    print(f"{tipo:<15}  diff_max={max_diff:.1f}  "
          f"participacion_min={df_sec['participacion'].min():.3f}  "
          f"participacion_max={df_sec['participacion'].max():.3f}")
    df_sec.drop(columns=["_suma_partidos", "_diff"], inplace=True)

gubernatura      diff_max=119.0  participacion_min=0.307  participacion_max=1.331
diputaciones     diff_max=48.0  participacion_min=0.397  participacion_max=1.024
ayuntamiento     diff_max=44.0  participacion_min=0.384  participacion_max=1.000


## 7. Exportar por tipo de elección

In [ ]:
for tipo, df_sec in secciones.items():
    salida = PROCESADOS / f"resultados_seccion_{tipo}.csv"
    df_sec.to_csv(salida, index=False, encoding="utf-8-sig")
    print(f"Guardado: {salida}  ({df_sec.shape[0]} secciones × {df_sec.shape[1]} cols)")